# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os
import json
from typing import List

from dotenv import load_dotenv
from pydantic import BaseModel, Field
import chromadb
from chromadb.utils import embedding_functions
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from lib.parsers import PydanticOutputParser
from lib.memory import  LongTermMemory , MemoryFragment
from lib.vector_db import VectorStoreManager 

In [3]:
# TODO: Load environment variables
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_functions.OpenAIEmbeddingFunction(),
)


@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    result = collection.query(
        query_texts=[query],
        n_results=5,
        include=["documents", "metadatas", "distances"],
    )

    games = []
    for metadata, distance in zip(result["metadatas"][0], result["distances"][0]):
        games.append({
            "Platform": metadata["Platform"],
            "Name": metadata["Name"],
            "YearOfRelease": metadata["YearOfRelease"],
            "Description": metadata["Description"],
            "distance": distance,
        })
    return games

#### Evaluate Retrieval Tool

In [5]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result


class EvaluationReport(BaseModel):
    """Structured output of the LLM judge"""
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Detailed explanation of the evaluation result")


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[dict]) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    # The LLM may pass the documents as a list or as an already-serialized string
    if isinstance(retrieved_docs, str):
        docs_text = retrieved_docs
    else:
        docs_text = json.dumps(retrieved_docs, ensure_ascii=False, indent=2, default=str)

    judge = LLM(model="gpt-4o-mini", temperature=0.0)

    messages = [
        SystemMessage(content=(
            "Your task is to evaluate if the documents are enough to respond the query. "
            "Give a detailed explanation, so it's possible to take an action to accept it or not."
        )),
        UserMessage(content=f"Query: {question}\n\nDocuments:\n{docs_text}"),
    ]

    response = judge.invoke(messages, response_format=EvaluationReport)
    report = PydanticOutputParser(model_class=EvaluationReport).parse(response)
    return report.model_dump()

#### Game Web Search Tool

In [6]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool
def game_web_search(question: str) -> list:
    """
    Web search: Finds the most relevant results on the web
    args:
    - question: a question about game industry.

    You'll receive results as list. Each element contains:
    - title: Title of the web page
    - url: URL of the web page
    - content: Relevant snippet of the web page
    """
    client = TavilyClient(api_key=TAVILY_API_KEY)

    response = client.search(
        query=question,
        max_results=5,
    )

    return [
        {"title": r["title"], "url": r["url"], "content": r["content"]}
        for r in response["results"]
    ]

### Agent

In [7]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

instructions = """
You are UdaPlay, an AI research assistant specialized in the video game industry.

You have access to these tools:
- retrieve_game: semantic search over the internal video game knowledge base (vector DB).
- evaluate_retrieval: an LLM judge that checks whether the retrieved documents are enough to answer the question.
- game_web_search: web search (Tavily) for information that is missing from the knowledge base.

Follow this workflow for EVERY question:
1. Call retrieve_game with the user's question.
2. Call evaluate_retrieval with the original question and the documents returned by retrieve_game.
3. If the evaluation says the documents are useful, answer using ONLY those documents.
   If not, call game_web_search and answer using the web results instead.

Rules:
- Never invent facts. Use only information returned by the tools.
- Always state where the answer came from: "internal knowledge base" or "web search" (include URLs for web results).
- If neither source has the answer, say clearly that you could not find it.
- Answer in a structured way: game name, platform(s), year of release, then a short explanation.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)

# Agent.__init__ already builds the StateMachine (agent.workflow):
# __entry__ -> message_prep -> llm_processor -> (tool_executor -> llm_processor)* -> __termination__
print(agent.workflow)
for transitions in agent.workflow.transitions.values():
    for transition in transitions:
        print(transition)

StateMachine(schema=['user_query', 'instructions', 'messages', 'current_tool_calls', 'total_tokens'])
Transition('__entry__' -> ['message_prep'])
Transition('message_prep' -> ['llm_processor'])
Transition('llm_processor' -> ['tool_executor', '__termination__'])
Transition('tool_executor' -> ['llm_processor'])


In [8]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

def print_run(run):
    """Show the tools called during this run and the final answer"""
    # messages present before this run (previous turns of the same session)
    previous_count = len(run.snapshots[0].state_data["messages"])
    final_state = run.get_final_state()
    new_messages = final_state["messages"][previous_count:]

    for msg in new_messages:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for call in msg.tool_calls:
                print(f"  [tool] {call.function.name}({call.function.arguments[:80]}...)")

    print(f"\nAnswer: {new_messages[-1].content}")
    print(f"Total tokens: {final_state.get('total_tokens', 0)}")


questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X realeased for Playstation 5?",
]

for question in questions:
    print("=" * 80)
    print(f"Question: {question}")
    run = agent.invoke(question)
    print_run(run)

Question: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


...

[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
  [tool] retrieve_game({"query":"Mortal Kombat X release for Playstation 5"}...)
  [tool] evaluate_retrieval({"question":"Was Mortal Kombat X released for Playstation 5?","retrieved_docs":[...)
  [tool] game_web_search({"question":"Was Mortal Kombat X released for Playstation 5?"}...)

Answer: **Game Name:** Mortal Kombat X  
**Platform(s):** PlayStation 4 (Playable on PlayStation 5)  
**Year of Release:** 2015  

Mortal Kombat X was originally released for PlayStation 4 and is playable on PlayStation 5, although some features available on PS4 may be absent. This information comes from the web search results ([PlayStation](https://www.playstation.com/en-us/games/mortal-kombat-x_msm_moved)).
Total tokens: 10761


### (Optional) Advanced

#### Long-term memory

In [9]:
# TODO: Update your agent with long-term memory
# LongTermMemory keeps MemoryFragments in a Chroma collection ("long_term_memory")
# embedded with OpenAI, so the agent can recall facts about a user across runs.
vector_db = VectorStoreManager(openai_api_key=OPENAI_API_KEY)
ltm = LongTermMemory(db=vector_db)

USER_ID = "user_123"

# Seed a few facts about this user (timestamp is generated automatically by MemoryFragment)
# Keep the default namespace: LongTermMemory.search() only looks in namespace="default" unless told otherwise
for content in [
    "User's favorite platform is the Nintendo Switch",
    "User loves platformer games, especially Mario titles",
    "User prefers short answers that always mention the year of release",
]:
    ltm.register(MemoryFragment(content=content, owner=USER_ID))

# Sanity check: semantic search scoped to this user
result = ltm.search("Which platform does the user like?", owner=USER_ID, limit=2)
for fragment, distance in zip(result.fragments, result.metadata["distances"]):
    print(f"{distance:.3f}  {fragment.content}")


0.118  User's favorite platform is the Nintendo Switch
0.140  User loves platformer games, especially Mario titles


#### Agent as a StateMachine with tools as pre-defined nodes

In [10]:
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes
from typing import Optional, TypedDict, Union
from lib.state_machine import StateMachine, Step, EntryPoint, Termination, Run
from lib.tooling import Tool


class GameAnswer(BaseModel):
    """Structured output returned by the agent"""
    answer: str = Field(description="Final answer to the user's question, in natural language")
    game_name: Optional[str] = Field(default=None, description="Name of the game the answer is about")
    platforms: List[str] = Field(default_factory=list, description="Platforms mentioned in the answer")
    year_of_release: Optional[int] = Field(default=None, description="Year of release, if known")


class UdaPlayState(TypedDict):
    user_query: str            # question from the user
    owner: str                 # user id, used to scope long-term memory
    memories: List[str]        # relevant long-term memories recalled for this user
    retrieved_docs: List[dict] # results of retrieve_game
    evaluation: dict           # result of evaluate_retrieval
    web_results: List[dict]    # results of game_web_search (only when retrieval is not useful)
    source: str                # "internal knowledge base" | "web search"
    answer: str                # final answer text
    structured_answer: dict    # GameAnswer as dict
    total_tokens: int          # cumulative token usage


class UdaPlayAgent:
    """
    UdaPlay as a StateMachine where every tool is a pre-defined node
    (the LLM does not choose tools, the graph does):

    __entry__ -> recall_memory -> retrieve_game -> evaluate_retrieval
                                                     ├─ useful     -> generate_answer
                                                     └─ not useful -> game_web_search -> generate_answer
    generate_answer -> store_memory -> __termination__
    """

    def __init__(self,
                 model_name: str,
                 instructions: str,
                 tools: List[Tool],
                 long_term_memory: LongTermMemory,
                 temperature: float = 0.0,
                 llm: Optional[LLM] = None):
        self.instructions = instructions
        self.tools = {t.name: t for t in tools}
        self.ltm = long_term_memory
        self.llm = llm or LLM(model=model_name, temperature=temperature)
        self.workflow = self._create_state_machine()

    # ---------- step logic ----------

    def _recall_memory_step(self, state: UdaPlayState) -> dict:
        """Look up what we already know about this user"""
        result = self.ltm.search(state["user_query"], owner=state["owner"])
        return {"memories": [fragment.content for fragment in result.fragments]}

    def _retrieve_game_step(self, state: UdaPlayState) -> dict:
        docs = self.tools["retrieve_game"](query=state["user_query"])
        return {"retrieved_docs": docs, "source": "internal knowledge base"}

    def _evaluate_retrieval_step(self, state: UdaPlayState) -> dict:
        report = self.tools["evaluate_retrieval"](
            question=state["user_query"],
            retrieved_docs=state["retrieved_docs"],
        )
        return {"evaluation": report}

    def _web_search_step(self, state: UdaPlayState) -> dict:
        results = self.tools["game_web_search"](question=state["user_query"])
        return {"web_results": results, "source": "web search"}

    def _generate_answer_step(self, state: UdaPlayState) -> dict:
        """Answer with the LLM using the chosen source + recalled memories, as structured output"""
        if state["source"] == "web search":
            context = state["web_results"]
        else:
            context = state["retrieved_docs"]

        memories = "\n".join(f"- {m}" for m in state["memories"]) or "- (nothing yet)"
        messages = [
            SystemMessage(content=self.instructions),
            UserMessage(content=(
                f"What you remember about this user:\n{memories}\n\n"
                f"Source of the context below: {state['source']}\n"
                f"Context:\n{json.dumps(context, ensure_ascii=False, indent=2, default=str)}\n\n"
                f"Question: {state['user_query']}"
            )),
        ]

        response = self.llm.invoke(messages, response_format=GameAnswer)
        parsed = PydanticOutputParser(model_class=GameAnswer).parse(response)

        total_tokens = state.get("total_tokens", 0)
        if response.token_usage:
            total_tokens += response.token_usage.total_tokens

        return {
            "answer": parsed.answer,
            "structured_answer": parsed.model_dump(),
            "total_tokens": total_tokens,
        }

    def _store_memory_step(self, state: UdaPlayState) -> dict:
        """Keep the Q/A in long-term memory so future runs can reuse it"""
        self.ltm.register(MemoryFragment(
            content=f"Q: {state['user_query']}\nA: {state['answer']}",
            owner=state["owner"],
        ))
        return {}

    # ---------- graph ----------

    def _create_state_machine(self) -> StateMachine[UdaPlayState]:
        machine = StateMachine[UdaPlayState](UdaPlayState)

        entry = EntryPoint[UdaPlayState]()
        recall_memory = Step[UdaPlayState]("recall_memory", self._recall_memory_step)
        retrieve_game = Step[UdaPlayState]("retrieve_game", self._retrieve_game_step)
        evaluate_retrieval = Step[UdaPlayState]("evaluate_retrieval", self._evaluate_retrieval_step)
        game_web_search = Step[UdaPlayState]("game_web_search", self._web_search_step)
        generate_answer = Step[UdaPlayState]("generate_answer", self._generate_answer_step)
        store_memory = Step[UdaPlayState]("store_memory", self._store_memory_step)
        termination = Termination[UdaPlayState]()

        machine.add_steps([
            entry, recall_memory, retrieve_game, evaluate_retrieval,
            game_web_search, generate_answer, store_memory, termination,
        ])

        machine.connect(entry, recall_memory)
        machine.connect(recall_memory, retrieve_game)
        machine.connect(retrieve_game, evaluate_retrieval)

        def is_retrieval_useful(state: UdaPlayState) -> Union[Step[UdaPlayState], str]:
            """Transition logic: answer from the knowledge base, or fall back to the web"""
            if state["evaluation"].get("useful"):
                return generate_answer
            return game_web_search

        machine.connect(evaluate_retrieval, [generate_answer, game_web_search], is_retrieval_useful)
        machine.connect(game_web_search, generate_answer)
        machine.connect(generate_answer, store_memory)
        machine.connect(store_memory, termination)

        return machine

    def invoke(self, query: str, owner: str = "default") -> Run:
        initial_state: UdaPlayState = {
            "user_query": query,
            "owner": owner,
            "memories": [],
            "retrieved_docs": [],
            "evaluation": {},
            "web_results": [],
            "source": "",
            "answer": "",
            "structured_answer": {},
            "total_tokens": 0,
        }
        return self.workflow.run(initial_state)


udaplay_agent = UdaPlayAgent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    long_term_memory=ltm,
    temperature=0.0,
)

print(udaplay_agent.workflow)
for transitions in udaplay_agent.workflow.transitions.values():
    for transition in transitions:
        print(transition)


StateMachine(schema=['user_query', 'owner', 'memories', 'retrieved_docs', 'evaluation', 'web_results', 'source', 'answer', 'structured_answer', 'total_tokens'])
Transition('__entry__' -> ['recall_memory'])
Transition('recall_memory' -> ['retrieve_game'])
Transition('retrieve_game' -> ['evaluate_retrieval'])
Transition('evaluate_retrieval' -> ['generate_answer', 'game_web_search'])
Transition('game_web_search' -> ['generate_answer'])
Transition('generate_answer' -> ['store_memory'])
Transition('store_memory' -> ['__termination__'])


In [11]:
# Invoke the state-machine agent: the same 3 questions, plus one that relies on long-term memory
def print_advanced_run(run):
    final_state = run.get_final_state()
    print("  path      :", " -> ".join(s.step_id for s in run.snapshots))
    print("  memories  :", final_state["memories"])
    print("  source    :", final_state["source"])
    print("  answer    :", final_state["answer"])
    print("  structured:", final_state["structured_answer"])
    print("  tokens    :", final_state["total_tokens"])


questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X realeased for Playstation 5?",
    # needs long-term memory: the favorite platform is never mentioned in the question
    "Is there a Mario Kart game in the knowledge base, and is it on my favorite platform?",
]

for question in questions:
    print("=" * 80)
    print(f"Question: {question}")
    run = udaplay_agent.invoke(question, owner=USER_ID)
    print_advanced_run(run)


Question: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__


[StateMachine] Executing step: recall_memory


[StateMachine] Executing step: retrieve_game


[StateMachine] Executing step: evaluate_retrieval


[StateMachine] Executing step: generate_answer


...

[StateMachine] Executing step: recall_memory


[StateMachine] Executing step: retrieve_game


[StateMachine] Executing step: evaluate_retrieval


[StateMachine] Executing step: generate_answer


[StateMachine] Executing step: store_memory
[StateMachine] Terminating: __termination__
  path      : __entry__ -> recall_memory -> retrieve_game -> evaluate_retrieval -> generate_answer -> store_memory
  memories  : ['User loves platformer games, especially Mario titles', "User's favorite platform is the Nintendo Switch", 'Q: Which one was the first 3D platformer Mario game?\nA: The first 3D platformer Mario game is Super Mario 64, released in 1996.']
  source    : internal knowledge base
  answer    : Yes, there is a Mario Kart game in the knowledge base. It is called Mario Kart 8 Deluxe, released in 2017, and it is available on your favorite platform, the Nintendo Switch.
  structured: {'answer': 'Yes, there is a Mario Kart game in the knowledge base. It is called Mario Kart 8 Deluxe, released in 2017, and it is available on your favorite platform, the Nintendo Switch.', 'game_name': 'Mario Kart 8 Deluxe', 'platforms': ['Nintendo Switch'], 'year_of_release': 2017}
  tokens    : 935
